# Power Infrastructure Density: NC to Static Feature Grid

**Purpose:** Convert the 5 power infrastructure density NetCDF files to flat
DataFrames, map each onto the weather reanalysis grid via a KD-Tree
nearest-neighbour lookup, and join them onto the static feature grid.

| Variable | File |
|----------|------|
| Minor line density | `minor_line_density.nc` |
| Pole density | `pole_density.nc` |
| Tower density | `tower_density.nc` |
| Transmission line density | `trans_line_density.nc` |
| Transformer density | `transformer_density.nc` |

**Pipeline Overview:**

| Phase | Steps |
|-------|-------|
| **A. Inspect** | Load and inspect each `.nc` file; check dimensions and coverage |
| **B. KD-Tree Match** | For each variable, nearest-neighbour match onto weather grid; geodesic QA |
| **C. Join & Save** | Left-join all density columns onto static grid; fill NA with 0; save |

**Inputs:**
- `Power_Data/*.nc` — 5 power infrastructure density NetCDF files
- `Clean_Data/Power_Data/transmission_line_density_match_weather_grid.parquet` — CEC transmission line density (pre-matched, from `00_04`)
- `Clean_Data/static_features.parquet` — base static grid (from `02_05`)

**Output:** `Clean_Data/static_features_add_openmap.parquet` — static grid + power density features

> **Note:** Missing values are filled with 0 — a cell with no nearby infrastructure
> has zero density by definition.

> **Run order:** `02_05` must run before this notebook.

## 0. Configuration

Centralized path configuration — **edit this cell only**.

In [1]:
import os

# ====================== EDIT THESE PATHS ======================
PROJECT_ROOT = r"E:\zcao\CA_Wildfire"

# Input: power density NetCDF files
POWER_DATA_DIR = os.path.join(PROJECT_ROOT, "Power_Data")

# Input: base static grid (created by 02_05_Build_-_Static_Features.ipynb)
# NOTE: 02_05 must run before this notebook
STATIC_FEATURES_PATH = os.path.join(PROJECT_ROOT, "Clean_Data",
                                     "static_features.parquet")

# Distance threshold — grid cells > this km from nearest power pixel set to 0
MAX_MATCH_KM = 4.0

# Output
OUTPUT_PATH = os.path.join(PROJECT_ROOT, "Clean_Data",
                           "static_features_add_openmap.parquet")

# Expected NC files
NC_FILES = [
    'minor_line_density.nc',
    'pole_density.nc',
    'tower_density.nc',
    'trans_line_density.nc',
    'transformer_density.nc',
]

# CEC transmission line density — pre-matched parquet from 00_04
# (NC → weather grid match done separately in 00_04_CEC_Transmission_Line_Density.ipynb)
TRANSMISSION_LINE_PATH = os.path.join(
    PROJECT_ROOT, "Clean_Data", "Power_Data",
    "transmission_line_density_match_weather_grid.parquet"
)

print(f"Power data dir   : {POWER_DATA_DIR}")
print(f"Static grid      : {STATIC_FEATURES_PATH}")
print(f"Output           : {OUTPUT_PATH}")
print(f"Match threshold  : {MAX_MATCH_KM} km")
for label, path in [("POWER_DATA_DIR",      POWER_DATA_DIR),
                    ("STATIC_FEATURES_PATH", STATIC_FEATURES_PATH)]:
    print(f"  [{'OK' if os.path.exists(path) else 'MISSING'}] {label}")

Power data dir   : E:\zcao\CA_Wildfire\Power_Data
Static grid      : E:\zcao\CA_Wildfire\Clean_Data\static_features.parquet
Output           : E:\zcao\CA_Wildfire\Clean_Data\static_features_add_openmap.parquet
Match threshold  : 4.0 km
  [OK] POWER_DATA_DIR
  [OK] STATIC_FEATURES_PATH


## 1. Environment Setup

In [2]:
import sys, gc, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import xarray as xr
from scipy.spatial import cKDTree
from geopy.distance import geodesic
from tqdm import tqdm

gc.collect()
print(f"Python : {sys.version.split('|')[0].strip()}")
print(f"pandas : {pd.__version__}")
print(f"xarray : {xr.__version__}")
print(f"numpy  : {np.__version__}")

Python : 3.9.13 (main, Aug 25 2022, 23:51:50) [MSC v.1916 64 bit (AMD64)]
pandas : 2.2.2
xarray : 2024.7.0
numpy  : 1.24.4


---

# Phase A: Inspect Power Density NetCDF Files

Load each `.nc` file, print dimensions and the variable name, and confirm
all files share the same spatial grid before matching.

## 2. Load Static Feature Grid

In [3]:
static = pd.read_parquet(STATIC_FEATURES_PATH)
weather_grid = static[['lon','lat']].drop_duplicates().reset_index(drop=True)

print(f"Static grid shape  : {static.shape}")
print(f"Static columns     : {list(static.columns)}")
print(f"Unique grid cells  : {len(weather_grid):,}")

Static grid shape  : (13048, 8)
Static columns     : ['lon', 'lat', 'fire_attribute', 'veg', 'slope_avg', 'slope_max', 'road_density_km_km2', 'SubRegion']
Unique grid cells  : 13,048


## 3. Inspect All NC Files

In [4]:
print(f"Files in Power_Data/:")
for f in sorted(os.listdir(POWER_DATA_DIR)):
    print(f"  {f}")

print()
ref_dims = None
for nc_file in NC_FILES:
    ds = xr.open_dataset(os.path.join(POWER_DATA_DIR, nc_file))
    var_name = [v for v in ds.data_vars][0]
    print(f"{nc_file:<30} dims={dict(ds.dims)}  variable='{var_name}'")
    if ref_dims is None:
        ref_dims = dict(ds.dims)
    else:
        if dict(ds.dims) != ref_dims:
            print(f"  WARNING: dims differ from reference {ref_dims}")
    ds.close()
print(f"\nAll files checked. Reference dims: {ref_dims}")

Files in Power_Data/:
  TransmissionLine_CEC.cpg
  TransmissionLine_CEC.dbf
  TransmissionLine_CEC.prj
  TransmissionLine_CEC.shp
  TransmissionLine_CEC.shp.xml
  TransmissionLine_CEC.shx
  minor_line_density.nc
  pole_density.nc
  tower_density.nc
  trans_line_density.nc
  transformer_density.nc
  transmission_line_density.CA.nc

minor_line_density.nc          dims={'lat': 238, 'lon': 258}  variable='minor_line_density'
pole_density.nc                dims={'lat': 238, 'lon': 258}  variable='pole_density'
tower_density.nc               dims={'lat': 238, 'lon': 258}  variable='tower_density'
trans_line_density.nc          dims={'lat': 238, 'lon': 258}  variable='trans_line_density'
transformer_density.nc         dims={'lat': 238, 'lon': 258}  variable='transformer_density'

All files checked. Reference dims: {'lat': 238, 'lon': 258}


---

# Phase B: KD-Tree Match onto Weather Grid

For each NetCDF file:
1. Flatten to DataFrame, drop NA rows
2. Build KD-Tree on power data coordinates
3. Query nearest neighbour for each weather grid cell
4. Compute exact geodesic distances
5. Apply `MAX_MATCH_KM` threshold — cells beyond threshold are set to 0

All 5 variables are matched in a single loop.

## 4. Match All Power Density Variables

In [5]:
grid_coords = weather_grid[['lat','lon']].values
density_frames = []  # collect one DataFrame per variable

for nc_file in NC_FILES:
    var_name = os.path.splitext(nc_file)[0]  # e.g. 'minor_line_density'
    print(f"\n{'='*55}")
    print(f"Processing: {nc_file}  →  column: '{var_name}'")

    # Load and flatten
    ds  = xr.open_dataset(os.path.join(POWER_DATA_DIR, nc_file))
    df  = ds.to_dataframe().reset_index().dropna(subset=[var_name])
    ds.close()
    print(f"  Raw rows (non-NA): {len(df):,}")

    # KD-Tree approximate match
    tree = cKDTree(df[['lat','lon']].values)
    approx_dist, indices = tree.query(grid_coords, k=1)

    # Exact geodesic distances
    exact_dist = np.empty(len(weather_grid))
    power_coords = df[['lat','lon']].values
    for i in tqdm(range(len(weather_grid)),
                  desc=f'  Geodesic ({var_name})', leave=False):
        exact_dist[i] = geodesic(
            (grid_coords[i,0], grid_coords[i,1]),
            (power_coords[indices[i],0], power_coords[indices[i],1])
        ).km

    mask = exact_dist < MAX_MATCH_KM
    matched_vals = np.where(mask, df.iloc[indices][var_name].values, 0.0)

    print(f"  Matched (< {MAX_MATCH_KM} km) : {mask.sum():,} / {len(mask):,} "
          f"({mask.mean()*100:.1f}%)")
    print(f"  Unmatched → 0              : {(~mask).sum():,}")
    print(f"  Distance range (km)        : {exact_dist.min():.3f} → "
          f"{exact_dist.max():.3f}")

    density_frames.append(
        weather_grid.copy().assign(**{var_name: matched_vals})
    )

    del df, tree, approx_dist, indices, exact_dist, mask, matched_vals
    gc.collect()

print(f"\nAll {len(density_frames)} variables processed.")


Processing: minor_line_density.nc  →  column: 'minor_line_density'
  Raw rows (non-NA): 24,589


  Matched (< 4.0 km) : 13,035 / 13,048 (99.9%)
  Unmatched → 0              : 13
  Distance range (km)        : 0.000 → 50.878

Processing: pole_density.nc  →  column: 'pole_density'
  Raw rows (non-NA): 24,589


  Matched (< 4.0 km) : 13,035 / 13,048 (99.9%)
  Unmatched → 0              : 13
  Distance range (km)        : 0.000 → 50.878

Processing: tower_density.nc  →  column: 'tower_density'
  Raw rows (non-NA): 24,589


  Matched (< 4.0 km) : 13,035 / 13,048 (99.9%)
  Unmatched → 0              : 13
  Distance range (km)        : 0.000 → 50.878

Processing: trans_line_density.nc  →  column: 'trans_line_density'
  Raw rows (non-NA): 24,589


  Matched (< 4.0 km) : 13,035 / 13,048 (99.9%)
  Unmatched → 0              : 13
  Distance range (km)        : 0.000 → 50.878

Processing: transformer_density.nc  →  column: 'transformer_density'
  Raw rows (non-NA): 24,589


  Matched (< 4.0 km) : 13,035 / 13,048 (99.9%)
  Unmatched → 0              : 13
  Distance range (km)        : 0.000 → 50.878

All 5 variables processed.


---

# Phase C: Join onto Static Grid & Save

Merge all matched density columns onto the static feature grid
by `(lon, lat)`. Any remaining NaN (edge cases) is filled with 0.

## 5. Merge Density Columns onto Static Grid

In [6]:
result = static.copy()
print(f"Starting shape: {result.shape}")

# Join the 5 NC-matched density columns
for frame in density_frames:
    density_col = [c for c in frame.columns if c not in ['lon','lat']][0]
    result = result.merge(frame, on=['lon','lat'], how='left')
    print(f"  + {density_col:<30} → shape: {result.shape}")

# Join CEC transmission line density (pre-matched parquet from 00_04)
trans_line = pd.read_parquet(TRANSMISSION_LINE_PATH)
result = result.merge(trans_line, on=['lon','lat'], how='left')
print(f"  + {'line_density_km_per_cell':<30} → shape: {result.shape}")

# Fill any residual NaN with 0
density_cols = [c for c in result.columns if c not in static.columns]
n_nan = result[density_cols].isnull().sum().sum()
if n_nan > 0:
    result[density_cols] = result[density_cols].fillna(0)
    print(f"Filled {n_nan} residual NaN values with 0")

print(f"\nFinal shape : {result.shape}")
print(f"Columns     : {list(result.columns)}")

Starting shape: (13048, 8)
  + minor_line_density             → shape: (13048, 9)
  + pole_density                   → shape: (13048, 10)
  + tower_density                  → shape: (13048, 11)
  + trans_line_density             → shape: (13048, 12)
  + transformer_density            → shape: (13048, 13)
  + line_density_km_per_cell       → shape: (13048, 14)

Final shape : (13048, 14)
Columns     : ['lon', 'lat', 'fire_attribute', 'veg', 'slope_avg', 'slope_max', 'road_density_km_km2', 'SubRegion', 'minor_line_density', 'pole_density', 'tower_density', 'trans_line_density', 'transformer_density', 'line_density_km_per_cell']


## 6. Save Output

In [7]:
result.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved -> {OUTPUT_PATH}")
print(f"File size  : {os.path.getsize(OUTPUT_PATH)/1e6:.1f} MB")
print(f"Final shape: {result.shape[0]:,} rows × {result.shape[1]} cols")

del result, density_frames, static, weather_grid
gc.collect()

Saved -> E:\zcao\CA_Wildfire\Clean_Data\static_features_add_openmap.parquet
File size  : 0.3 MB
Final shape: 13,048 rows × 14 cols


0

## 7. Summary

| Phase | Step | Description | Key Result |
|-------|------|-------------|------------|
| A | Load static grid | `static_features.parquet` | Base (lon, lat) grid |
| A | Inspect NC files | Check dims + variable names for all 5 files | Grid consistency confirmed |
| B | KD-Tree match | Nearest power pixel per weather grid cell | Fast approximate match |
| B | Geodesic QA | Exact km distances | Cells > 4 km → 0 |
| B | Assign values | `np.where(mask, value, 0)` | 5 NC density columns |
| C | Join | Merge 5 NC frames + 1 parquet onto static grid | 6 new columns |
| C | Fill NA | Residual NaN → 0 | No NaN in output |
| C | Save | `static_features_add_openmap.parquet` | Input for `03_04` |